## An extra exercise for those who enjoy web scraping

You may notice that if you try `display_summary("https://openai.com")` - it doesn't work! That's because OpenAI has a fancy website that uses Javascript. There are many ways around this that some of you might be familiar with. For example, Selenium is a hugely popular framework that runs a browser behind the scenes, renders the page, and allows you to query it. If you have experience with Selenium, Playwright or similar, then feel free to improve the Website class to use them. In the community-contributions folder, you'll find an example Selenium solution from a student (thank you!)

In [ ]:
import os
from dotenv import load_dotenv

from IPython.display import Markdown, display
from openai import OpenAI

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

class TinyScraper:
    def __init__(self, headless=False):
        options = Options()
        if headless:
            options.add_argument("--headless=new")

        self.driver = webdriver.Chrome(options=options)

    def scrape(self, url):
        self.driver.get(url)

        title = self.driver.title

        # Recursive DOM text extraction via JS
        content = self.driver.execute_script("""
            function getText(node) {
                let text = "";

                // Skip script/style/noscript
                if (node.nodeName === "SCRIPT" || 
                    node.nodeName === "STYLE" || 
                    node.nodeName === "NOSCRIPT") {
                    return "";
                }

                // If text node
                if (node.nodeType === Node.TEXT_NODE) {
                    return node.nodeValue.trim() + " ";
                }

                // Recurse children
                for (let child of node.childNodes) {
                    text += getText(child);
                }

                return text;
            }

            return getText(document.body);
        """)

        return {
            "url": url,
            "title": title,
            "content": content[:2000]
        }

    def close(self):
        self.driver.quit()

# Load environment variables in a file called .env
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')


# Check the key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

openai = OpenAI()

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."


def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website['title']}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website['text']
    return user_prompt


def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

def summarize(url):
    scraper = TinyScraper()
    data = scraper.scrape("https://openai.com")
    scraper.close()
    website = {
        "url": data["url"],
        "title": data["title"],
        "text": data["content"]
    }    
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

display_summary("https://openai.com")